# 🎯 In-Class Lab: K-Nearest Neighbors — 100% from scratch
Everything in this notebook is built by hand (numpy for arrays, matplotlib for plots — no ML library).
Only the **very last section** shows the library version, so you can compare.

**Rules of the lab:** before running a "🤔 PREDICT FIRST" cell, write your guess in the empty line — then run and check.

> ✏️ **Practice version** — some lines are replaced with `____`. Fill them in using the clue comments (🖊️ FILL IN), then run the cell to check yourself. The completed notebook is shared after class.

## 📖 Theory recall (2 min, before you code)
- **Instance-based ("lazy") learning** — KNN has *no training phase*: it memorizes the data and does all the work at prediction time.
- **Distance** — Euclidean: $d(p,q)=\sqrt{\sum_j (p_j-q_j)^2}$, the straight-line distance between two points.
- **Prediction** — find the $k$ closest training points, then take a **majority vote** of their labels.
- **The role of k** — small $k$: flexible, noisy boundary (**overfitting**); large $k$: smooth boundary that drifts toward the global majority (**underfitting**).
- **Features must share a scale** — a feature in big units dominates the distance and drowns the others (the trap in section 5).
- **Cost** — every query computes $n$ distances over $d$ features: $O(n \cdot d)$ per prediction.

## 0. Even the DATA is from scratch
Two interleaved half-circles ("moons") — a shape that will matter later.

In [ ]:
import numpy as np, matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap

def make_moons_scratch(n=200, noise=0.30, seed=0):
    rng = np.random.default_rng(seed)              # seeded generator -> everyone gets the same "random" data
    n2 = n // 2
    t = np.linspace(0, np.pi, n2)                  # angles along a half-circle
    outer = np.c_[np.cos(t), np.sin(t)]            # class 0: points on the upper half-circle
    inner = np.c_[1 - np.cos(t), 0.5 - np.sin(t)]  # class 1: the same arc flipped and shifted -> lower moon
    X = np.vstack([outer, inner]) + rng.normal(0, noise, (n, 2))   # stack both moons, add Gaussian jitter
    y = np.array([0]*n2 + [1]*n2)                  # first half is class 0, second half class 1
    idx = rng.permutation(n)                       # shuffle so the classes are mixed
    return X[idx], y[idx]

def train_test_split_scratch(X, y, test_frac=0.3, seed=0):
    rng = np.random.default_rng(seed)
    idx = rng.permutation(len(X))                  # random order of all row indices
    cut = int(len(X) * (1 - test_frac))            # where train ends and test begins
    tr, te = idx[:cut], idx[cut:]
    return X[tr], X[te], y[tr], y[te]

X, y = make_moons_scratch()
X_train, X_test, y_train, y_test = train_test_split_scratch(X, y)

plt.figure(figsize=(6,4))
plt.scatter(X_train[:,0], X_train[:,1], c=y_train, cmap="coolwarm", edgecolor="k")
plt.title("Training data — two classes (handmade!)"); plt.show()

## 1. KNN in 12 lines — the WHOLE algorithm
Read it line by line: distance → sort → take k → vote. That's all KNN is.

In [1]:
def knn_predict(X_train, y_train, x, k=3):
    dists = dists = np.sqrt(((X_train - x) ** 2).sum(axis=1))                                   # 🖊️ 1. FILL IN: Euclidean distance from x to EVERY training point (hint: subtract, square, .sum(axis=1), np.sqrt)
    nearest = nearest = np.argsort(dists)[:k]                                  # 🖊️ 2. FILL IN: indices of the k CLOSEST points (hint: np.argsort, then slice [:k])
    votes = y_train[nearest]                          # 3. look up the labels of those k neighbors
    return np.bincount(votes).argmax()                                     # 🖊️ 4. FILL IN: the label with the most votes (hint: np.bincount(votes).argmax())

# sanity check on one point
print("prediction:", knn_predict(X_train, y_train, np.array([0.0, 0.5]), k=5))

NameError: name 'X_train' is not defined

### ✍️ Trace it BY HAND (2 min, with your neighbor)
For the query x = (0, 0.5) and k = 5: which of the 4 lines does the most "work"?
If we have n training points and d features, how many subtractions does line 1 perform? `____`

## 2. 🤔 PREDICT FIRST
The point ⭐ = (1.5, -0.5) sits in the lower-right area. **Write your guess (0=blue or 1=red) here:** `____`

Then run the cell.

In [ ]:
star = np.array([1.5, -0.5])
plt.figure(figsize=(6,4))
plt.scatter(X_train[:,0], X_train[:,1], c=y_train, cmap="coolwarm", edgecolor="k", alpha=0.6)
plt.scatter(*star, marker="*", s=400, c="gold", edgecolor="k")
plt.title("Where does the ⭐ belong?"); plt.show()

# let's SEE the vote, not just the answer:
dists = np.sqrt(((X_train - star)**2).sum(axis=1))
nearest = np.argsort(dists)[:5]
for i in nearest:
    print(f"neighbor at ({X_train[i,0]:+.2f}, {X_train[i,1]:+.2f})  distance {dists[i]:.2f}  votes class {y_train[i]}")
print("majority ->", knn_predict(X_train, y_train, star, k=5))

## 3. Play with k — the interactive part
Drag the slider. **Watch what happens to the boundary at k=1 vs k=101.**

In [ ]:
from ipywidgets import interact, IntSlider

def plot_boundary(k=5):
    xx, yy = np.meshgrid(np.linspace(-2, 3, 120), np.linspace(-1.5, 2, 120))
    grid = np.c_[xx.ravel(), yy.ravel()]
    Z = np.array([knn_predict(X_train, y_train, p, k) for p in grid]).reshape(xx.shape)
    plt.figure(figsize=(6,4))
    plt.contourf(xx, yy, Z, cmap=ListedColormap(["#aec7e8", "#ff9896"]), alpha=0.7)
    plt.scatter(X_train[:,0], X_train[:,1], c=y_train, cmap="coolwarm", edgecolor="k", s=25)
    acc = np.mean([knn_predict(X_train, y_train, p, k) for p in X_test] == y_test)
    plt.title(f"k={k}   test accuracy={acc:.2f}"); plt.show()

interact(plot_boundary, k=IntSlider(min=1, max=101, step=2, value=5));

### 💬 Discuss with your neighbor (2 min)
1. Why does **k=1** produce those little "islands"? What is that phenomenon called?
2. What happens at **k=101** and why? (Hint: how many training points are there?)

## 4. ⚔️ Mini-challenge (5 min): find the best k
Fill in the loop, then shout your best (k, accuracy) for the class leaderboard.

In [ ]:
best_k, best_acc = None, 0
for k in range(1, 60, 2):                             # odd k only, so votes can't tie 50/50
    acc = ____                                        # 🖊️ FILL IN: test accuracy for this k (hint: same pattern as the slider cell above)
    if ____:                                          # 🖊️ FILL IN: when is this k the new champion?
        best_k, best_acc = k, acc
print(f"🏆 best k={best_k}  accuracy={best_acc:.3f}")

## 5. The scaling trap 🪤 — 🤔 PREDICT FIRST
We multiply **feature 2 only** by 100 (imagine: cm → money in rupiah).
**Will accuracy go up / down / stay the same?** Your guess: `____`

In [ ]:
X_bad_train, X_bad_test = X_train.copy(), X_test.copy()
X_bad_train[:,1] *= 100; X_bad_test[:,1] *= 100
acc_bad = np.mean([knn_predict(X_bad_train, y_train, p, 5) for p in X_bad_test] == y_test)
acc_ok  = np.mean([knn_predict(X_train, y_train, p, 5) for p in X_test] == y_test)
print(f"normal accuracy: {acc_ok:.2f}   after scaling one feature x100: {acc_bad:.2f}")
print("👉 distances are dominated by the big feature — ALWAYS scale features for KNN!")

---
## 6. 📚 NOW the library version — 3 lines
You now know exactly what happens inside these 3 lines. The library adds
smart data structures (KD-trees/ball-trees) so the neighbor search is much faster.

In [ ]:
import time
from sklearn.neighbors import KNeighborsClassifier

clf = KNeighborsClassifier(n_neighbors=5).fit(X_train, y_train)
lib_pred = clf.predict(X_test)
our_pred = np.array([knn_predict(X_train, y_train, p, 5) for p in X_test])
print("library and our scratch version agree on", np.mean(lib_pred == our_pred)*100, "% of test points")

# speed race on the 120x120 boundary grid (14,400 queries)
xx, yy = np.meshgrid(np.linspace(-2, 3, 120), np.linspace(-1.5, 2, 120))
grid = np.c_[xx.ravel(), yy.ravel()]
t0 = time.time(); _ = [knn_predict(X_train, y_train, p, 5) for p in grid]; t_scratch = time.time() - t0
t0 = time.time(); _ = clf.predict(grid);                                   t_lib = time.time() - t0
print(f"scratch: {t_scratch:.2f}s   sklearn: {t_lib:.3f}s   ({t_scratch/t_lib:.0f}x faster)")
print("same algorithm, same answers — engineering makes it fast. That's what libraries are for.")